# Proposições - Câmara/Brasil

Cada arquivo anual é baixado uma vez e separado para as 27 UFs.

Além dos dois CSVs principais, esta versão gera `proposicoes_fallback_YYYY.csv`. Ele só recebe dados quando um ID aparece em `proposicoesAutores`, mas não aparece no arquivo anual de proposições. Nesses casos o detalhe é buscado pela API.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import shutil
import tempfile
import time

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

CATALOGO = "workspace"
SCHEMA = "pi_ii_bronze"
VOLUME = "camara"
ID_LEGISLATURA = 57
ANOS = [2023, 2024, 2025, 2026]

UFS = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]

ROOT = Path(f"/Volumes/{CATALOGO}/{SCHEMA}/{VOLUME}")
# diretório temporário exclusivo desta execução
# evita conflito de permissão em retry/serverless
TMP = Path(tempfile.mkdtemp(prefix="pi_ii_bronze_camara_brasil_"))

# Se False, anos que já estiverem completos são reaproveitados.
FORCAR_RECOLETA = False

retry = Retry(
    total=6,
    connect=6,
    read=6,
    status=6,
    backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
)

session = requests.Session()
session.headers.update({
    "Accept": "*/*",
    "User-Agent": "PI-II-Univesp-Bronze-Camara-Brasil/1.0",
})
session.mount("https://", HTTPAdapter(max_retries=retry))
session.mount("http://", HTTPAdapter(max_retries=retry))


BASE_URL = "https://dadosabertos.camara.leg.br/api/v2"

def api_get(path, params=None):
    url = path if str(path).startswith("http") else BASE_URL + path
    r = session.get(url, params=params, timeout=(30, 120))
    r.raise_for_status()
    return r.json()

def pasta_uf(uf):
    return ROOT / uf.lower()

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def nome_normalizado(valor):
    return re.sub(r"[^a-z0-9]", "", str(valor).lower())

def encontrar_coluna(df, candidatos, contem_todos=None):
    mapa = {nome_normalizado(c): c for c in df.columns}

    for nome in candidatos:
        chave = nome_normalizado(nome)
        if chave in mapa:
            return mapa[chave]

    if contem_todos:
        termos = [nome_normalizado(x) for x in contem_todos]
        for c in df.columns:
            nc = nome_normalizado(c)
            if all(t in nc for t in termos):
                return c

    return None

def baixar(url, destino, timeout=(30, 900)):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)

    parcial = destino.with_suffix(destino.suffix + ".part")
    if parcial.exists():
        parcial.unlink()

    print("Baixando:", url)

    with session.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()

        total = int(r.headers.get("content-length") or 0)
        recebido = 0
        ultimo_print = time.time()

        with parcial.open("wb") as f:
            for bloco in r.iter_content(chunk_size=1024 * 1024):
                if not bloco:
                    continue

                f.write(bloco)
                recebido += len(bloco)

                if time.time() - ultimo_print >= 5:
                    if total:
                        print(
                            f"  {recebido / 1024**2:.1f} MB / "
                            f"{total / 1024**2:.1f} MB"
                        )
                    else:
                        print(f"  {recebido / 1024**2:.1f} MB")
                    ultimo_print = time.time()

    parcial.replace(destino)
    print(f"Concluído: {destino.name} ({destino.stat().st_size / 1024**2:.1f} MB)")
    return destino

def ler_csv_chunks(path, chunksize=100_000):
    return pd.read_csv(
        path,
        sep=";",
        encoding="utf-8",
        dtype=str,
        chunksize=chunksize,
        keep_default_na=False,
        na_filter=False,
    )

def append_local(df, path, cabecalho):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        path,
        sep=";",
        index=False,
        encoding="utf-8",
        mode="a",
        header=cabecalho,
    )

def copiar_para_volume(origem, destino):
    origem = Path(origem)
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)

    with origem.open("rb") as src, destino.open("wb") as dst:
        shutil.copyfileobj(src, dst, length=1024 * 1024 * 8)

def criar_csv_vazio(path, colunas):
    pd.DataFrame(columns=colunas).to_csv(
        path,
        sep=";",
        index=False,
        encoding="utf-8",
    )

def limpar_tmp(prefixo):
    pasta = TMP / prefixo
    pasta.mkdir(parents=True, exist_ok=True)
    return pasta

def carregar_deputados_brasil():
    partes = []

    for uf in UFS:
        path = pasta_uf(uf) / "deputados.csv"
        if not path.exists():
            raise FileNotFoundError(
                f"{path} não existe. Execute primeiro o notebook 00."
            )

        df = pd.read_csv(
            path,
            sep=";",
            dtype=str,
            keep_default_na=False,
        )
        partes.append(df)

    deputados = pd.concat(partes, ignore_index=True).drop_duplicates()

    id_col = encontrar_coluna(
        deputados,
        ["id", "idDeputado", "deputado_id"],
    )
    uf_col = encontrar_coluna(
        deputados,
        ["siglaUf", "uf", "deputado_siglaUf"],
    )

    if not id_col or not uf_col:
        raise RuntimeError(
            "Não encontrei as colunas de ID/UF em deputados.csv."
        )

    mapa = (
        deputados[[id_col, uf_col]]
        .assign(
            **{
                id_col: deputados[id_col].astype(str).str.strip(),
                uf_col: deputados[uf_col].astype(str).str.strip().str.upper(),
            }
        )
        .drop_duplicates(subset=[id_col])
        .set_index(id_col)[uf_col]
        .to_dict()
    )

    return deputados, mapa

def ano_completo(arquivos):
    return all(Path(x).exists() for x in arquivos)

def salvar_manifesto(nome, payload):
    payload = dict(payload)
    payload["gerado_em_utc"] = utc_now()

    path = ROOT / f"_manifest_{nome}_brasil.json"
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, default=str)

    print("Manifesto:", path)

print("Bronze Câmara:", ROOT)
print("UFs:", len(UFS))
print("Anos:", ANOS)


def marker_v3(nome, ano):
    return ROOT / f"_{nome}_v3_{ano}.ok"


In [ ]:
# mapa deputado -> UF
deputados, dep_para_uf = carregar_deputados_brasil()
print("Deputados no mapa:", len(dep_para_uf))


In [ ]:
def preparar_arquivos_tmp(pasta_tmp, prefixo, ano):
    return {
        uf: pasta_tmp / f"{prefixo}_{uf.lower()}_{ano}.csv"
        for uf in UFS
    }

def finalizar_split(arquivos_tmp, nome_saida, ano, colunas, contagens):
    for uf in UFS:
        local = arquivos_tmp[uf]
        destino = pasta_uf(uf) / f"{nome_saida}_{ano}.csv"

        if local.exists():
            copiar_para_volume(local, destino)
        else:
            criar_csv_vazio(destino, colunas)

        print(f"{uf}: {contagens.get(uf, 0):,} -> {destino.name}")


In [ ]:
resumo = []

for ano in ANOS:
    print("\n" + "=" * 70)
    print("ANO", ano)

    marker = marker_v3("proposicoes", ano)

    esperados = []
    for uf in UFS:
        esperados += [
            pasta_uf(uf) / f"proposicoes_autores_{ano}.csv",
            pasta_uf(uf) / f"proposicoes_{ano}.csv",
            pasta_uf(uf) / f"proposicoes_fallback_{ano}.csv",
        ]

    if (
        not FORCAR_RECOLETA
        and marker.exists()
        and ano_completo(esperados)
    ):
        print("Ano já revisado pela v3; pulando.")
        resumo.append({"ano": ano, "status": "ja_existia_v3"})
        continue

    pasta_tmp = limpar_tmp(f"proposicoes_{ano}")

    # autores
    url_autores = (
        "https://dadosabertos.camara.leg.br/arquivos/"
        f"proposicoesAutores/csv/proposicoesAutores-{ano}.csv"
    )
    csv_autores = pasta_tmp / f"proposicoesAutores-{ano}.csv"
    baixar(url_autores, csv_autores)

    tmp_autores = preparar_arquivos_tmp(
        pasta_tmp, "proposicoes_autores", ano
    )

    prop_ids_por_uf = {uf: set() for uf in UFS}
    cont_autores = {uf: 0 for uf in UFS}
    header_autores = {uf: True for uf in UFS}
    colunas_autores = None

    for chunk in ler_csv_chunks(csv_autores):
        if colunas_autores is None:
            colunas_autores = list(chunk.columns)

        uri_col = encontrar_coluna(
            chunk,
            ["uriAutor", "autor_uri", "uri_autor"],
            contem_todos=["uri", "autor"],
        )
        id_prop_col = encontrar_coluna(
            chunk,
            ["idProposicao", "proposicao_id", "id_proposicao"],
            contem_todos=["id", "proposicao"],
        )

        if not uri_col or not id_prop_col:
            raise RuntimeError(
                "Não encontrei uriAutor/idProposicao em proposicoesAutores. "
                f"Colunas: {list(chunk.columns)}"
            )

        dep_ids = (
            chunk[uri_col]
            .astype(str)
            .str.extract(r"/deputados/(\d+)(?:/)?$", expand=False)
            .fillna("")
        )

        ufs = dep_ids.map(dep_para_uf).fillna("")
        recorte = chunk.loc[ufs.isin(UFS)].copy()
        recorte["_uf_recorte"] = ufs.loc[recorte.index].values

        for uf, grupo in recorte.groupby("_uf_recorte"):
            if uf not in UFS:
                continue

            grupo = grupo.drop(columns=["_uf_recorte"])
            append_local(
                grupo,
                tmp_autores[uf],
                cabecalho=header_autores[uf],
            )
            header_autores[uf] = False
            cont_autores[uf] += len(grupo)

            prop_ids_por_uf[uf].update(
                grupo[id_prop_col].astype(str).str.strip()
            )

    finalizar_split(
        tmp_autores,
        "proposicoes_autores",
        ano,
        colunas_autores or [],
        cont_autores,
    )

    # proposições do arquivo anual
    url_props = (
        "https://dadosabertos.camara.leg.br/arquivos/"
        f"proposicoes/csv/proposicoes-{ano}.csv"
    )
    csv_props = pasta_tmp / f"proposicoes-{ano}.csv"
    baixar(url_props, csv_props)

    tmp_props = preparar_arquivos_tmp(
        pasta_tmp, "proposicoes", ano
    )
    cont_props = {uf: 0 for uf in UFS}
    header_props = {uf: True for uf in UFS}
    encontrados_por_uf = {uf: set() for uf in UFS}
    colunas_props = None

    for chunk in ler_csv_chunks(csv_props):
        if colunas_props is None:
            colunas_props = list(chunk.columns)

        id_col = encontrar_coluna(
            chunk,
            ["id", "idProposicao", "proposicao_id"],
        )
        if not id_col:
            raise RuntimeError(
                "Não encontrei o ID no arquivo de proposições. "
                f"Colunas: {list(chunk.columns)}"
            )

        ids_chunk = chunk[id_col].astype(str).str.strip()

        for uf in UFS:
            ids_uf = prop_ids_por_uf[uf]
            if not ids_uf:
                continue

            grupo = chunk.loc[ids_chunk.isin(ids_uf)].copy()
            if grupo.empty:
                continue

            append_local(
                grupo,
                tmp_props[uf],
                cabecalho=header_props[uf],
            )
            header_props[uf] = False
            cont_props[uf] += len(grupo)
            encontrados_por_uf[uf].update(
                grupo[id_col].astype(str).str.strip()
            )

    finalizar_split(
        tmp_props,
        "proposicoes",
        ano,
        colunas_props or [],
        cont_props,
    )

    # IDs que estão em autores, mas não no CSV anual
    faltantes_por_uf = {
        uf: sorted(prop_ids_por_uf[uf] - encontrados_por_uf[uf])
        for uf in UFS
    }
    todos_faltantes = sorted({
        prop_id
        for ids in faltantes_por_uf.values()
        for prop_id in ids
        if prop_id
    })

    print("IDs sem detalhe no CSV anual:", len(todos_faltantes))

    detalhes_api = {}
    erros_api = {}

    for i, prop_id in enumerate(todos_faltantes, start=1):
        try:
            payload = api_get(f"/proposicoes/{prop_id}")
            dado = payload.get("dados")

            if isinstance(dado, list):
                dado = dado[0] if dado else None

            if isinstance(dado, dict):
                detalhes_api[prop_id] = dado
            else:
                erros_api[prop_id] = "resposta sem objeto de dados"
        except Exception as exc:
            erros_api[prop_id] = f"{type(exc).__name__}: {exc}"

        if i % 20 == 0:
            print(f"Fallback API: {i}/{len(todos_faltantes)}")

    cont_fallback = {}

    for uf in UFS:
        rows = []

        for prop_id in faltantes_por_uf[uf]:
            dado = detalhes_api.get(prop_id)
            if not dado:
                continue

            row = pd.json_normalize([dado], sep="_")
            row["_id_proposicao"] = prop_id
            row["_uf_recorte"] = uf
            row["_ano_arquivo_autores"] = ano
            rows.append(row)

        if rows:
            df_fallback = pd.concat(rows, ignore_index=True, sort=False)
        else:
            df_fallback = pd.DataFrame(
                columns=[
                    "_id_proposicao",
                    "_uf_recorte",
                    "_ano_arquivo_autores",
                ]
            )

        destino = pasta_uf(uf) / f"proposicoes_fallback_{ano}.csv"
        df_fallback.to_csv(
            destino,
            sep=";",
            index=False,
            encoding="utf-8",
        )
        cont_fallback[uf] = len(df_fallback)

    auditoria = {
        "ano": ano,
        "ids_faltantes_unicos": len(todos_faltantes),
        "detalhes_recuperados_api": len(detalhes_api),
        "falhas_api": erros_api,
        "faltantes_por_uf": {
            uf: len(ids)
            for uf, ids in faltantes_por_uf.items()
        },
        "fallback_salvo_por_uf": cont_fallback,
    }

    with (ROOT / f"_auditoria_proposicoes_{ano}.json").open(
        "w", encoding="utf-8"
    ) as f:
        json.dump(auditoria, f, ensure_ascii=False, indent=2)

    marker.write_text(utc_now(), encoding="utf-8")

    resumo.append({
        "ano": ano,
        "status": "ok",
        "autores_total_recorte": sum(cont_autores.values()),
        "proposicoes_total_por_uf": sum(cont_props.values()),
        "ids_faltantes_unicos": len(todos_faltantes),
        "fallback_recuperado": len(detalhes_api),
        "fallback_falhas": len(erros_api),
    })

salvar_manifesto("proposicoes", {"resumo": resumo})

if "display" in globals():
    display(pd.DataFrame(resumo))
else:
    print(pd.DataFrame(resumo).to_string(index=False))
